# 01 Data Collection and Cleaning

?????????????????-??????????

???? GitHub ??????????????????????????


## Source: get_data.ipynb


In [ ]:
#爬取小学古诗文
import requests
from bs4 import BeautifulSoup
import json
import time
import re

# 基础URL
base_url = "https://www.gushiwen.cn"
list_url = f"{base_url}/gushi/xiaoxue.aspx"
headers = {"User-Agent": "Mozilla/5.0"}

def get_poem_links():
    """
    获取小学古诗中每首诗的链接
    """
    response = requests.get(list_url, headers=headers)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith("/shiwenv"):
            links.append(base_url + href)
    return list(set(links))

def get_poem_data(poem_url):
    """
    获取每首诗的详细信息：标题、朝代、作者、正文、译文
    """
    response = requests.get(poem_url, headers=headers)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, "html.parser")

    # 提取标题
    title_tag = soup.find("h1")
    title = title_tag.text.strip() if title_tag else ""

    # 提取朝代和作者信息
    source = soup.find("p", class_="source")
    dynasty = ""
    author = ""
    if source:
        a_tags = source.find_all("a")
        if len(a_tags) >= 2:
            dynasty = a_tags[1].text.strip()  # 朝代（在后）
            author = a_tags[0].text.strip()  # 作者（在前）

    # 去掉朝代的括号
    dynasty = re.sub(r"[〔〕]", "", dynasty)

    # 提取正文内容
    content_div = soup.find("div", class_="contson")
    content = content_div.get_text(separator="\n").strip() if content_div else ""

    # 提取译文（如有）
    translation = ""
    translation_div = soup.find("div", class_="contyishang")
    if translation_div:
        translation = translation_div.get_text(separator="\n").strip()

    return {
        "title": title,
        "dynasty": dynasty,
        "author": author,
        "content": content,
        "translation": translation
    }

def main():
    poem_links = get_poem_links()
    print(f"共找到 {len(poem_links)} 首小学古诗的链接。")

    all_poems = []
    for index, link in enumerate(poem_links, 1):
        try:
            data = get_poem_data(link)
            all_poems.append(data)
            print(f"({index}/{len(poem_links)}) 爬取：{data['title']} - 翻译：{'有' if data['translation'] else '无'}")
            time.sleep(1)  # 避免访问过快
        except Exception as e:
            print(f"爬取出错：{link}，错误信息：{e}")

    # 保存为 JSON 文件
    with open("xiaoxue_poems.json", "w", encoding="utf-8") as f:
        json.dump(all_poems, f, ensure_ascii=False, indent=2)

    print("爬取完成，数据已保存为 xiaoxue_poems.json。")

if __name__ == "__main__":
    main()


In [ ]:
#清洗小学古诗文
import re
import json

def clean_content(text):
    """
    对 content 字段进行清洗：
    1. 删除全角括号（……）和半角括号 (...) 内的内容；
    2. 按句号、叹号、冒号和换行符分句，去除空白句子；
    3. 将分句后的内容合并成一个连续字符串。
    """
    # 删除全角括号内的内容
    text = re.sub(r"（[^）]*）", "", text)
    # 删除半角括号内的内容
    text = re.sub(r"\([^)]*\)", "", text)
    
    # 按中文标点分句
    sentences = re.split(r'[。！？：\n]', text)
    # 去除空白句子
    sentences = [s.strip() for s in sentences if s.strip()]
    return "".join(sentences)

def clean_translation(text):
    """
    对 translation 字段进行清洗：
    1. 使用正则删除前缀 "译文及注释\n\n\n\n\n\n\n译文\n"（允许换行和空白）；
    2. 如果存在“注释”，则删除“注释”及其之后的所有内容；
    3. 按句号、叹号、冒号和换行符分句，去除空白句子；
    4. 将分句后的内容合并成一个连续字符串。
    """
    # 删除前缀
    prefix_pattern = r"^译文及注释\s*\n+\s*译文\s*\n+"
    text = re.sub(prefix_pattern, "", text)
    
    # 如果存在“注释”，则只保留“注释”之前的内容
    if "注释" in text:
        text = text.split("注释")[0]
    
    # 分句处理
    sentences = re.split(r'[。！？：\n]', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return "".join(sentences)

def contains_english(value):
    """
    判断传入的字符串或字符串列表中是否包含英文字母
    """
    if isinstance(value, str):
        return bool(re.search(r'[A-Za-z]', value))
    elif isinstance(value, list):
        for item in value:
            if re.search(r'[A-Za-z]', item):
                return True
        return False
    return False

def main():
    input_file = "xiaoxue_poems.json"
    output_file = "xiaoxue_poems_clean.json"
    
    # 读取原始数据
    with open(input_file, "r", encoding="utf-8") as f:
        poems = json.load(f)
    
    original_count = len(poems)
    cleaned_poems = []
    
    for poem in poems:
        # 对 content 和 translation 先进行清洗处理
        if "content" in poem:
            poem["content"] = clean_content(poem["content"])
        if "translation" in poem:
            poem["translation"] = clean_translation(poem["translation"])
        
        # 清洗后判断是否存在“展开阅读全文”，若存在则跳过该记录
        skip = False
        for field in ["title", "dynasty", "author", "content", "translation"]:
            if field in poem and re.search("展开阅读全文", poem[field]):
                skip = True
                break
        if skip:
            continue
        
        # 检查 title、dynasty、author、content、translation 五项中是否含有英文字符
        for field in ["title", "dynasty", "author", "content", "translation"]:
            if field in poem and contains_english(poem[field]):
                skip = True
                break
        if skip:
            continue
        
        cleaned_poems.append(poem)
    
    cleaned_count = len(cleaned_poems)
    
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(cleaned_poems, f, ensure_ascii=False, indent=4)
    
    print(f"原始数据数量: {original_count}")
    print(f"清洗后数据数量: {cleaned_count}")

if __name__ == "__main__":
    main()


In [ ]:
#爬取初中古诗文
import requests
from bs4 import BeautifulSoup
import json
import time
import re

# 基础URL
base_url = "https://www.gushiwen.cn"
list_url = f"{base_url}/gushi/chuzhong.aspx"
headers = {"User-Agent": "Mozilla/5.0"}

def get_poem_links():
    """
    获取初中古诗中每首诗的链接
    """
    response = requests.get(list_url, headers=headers)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith("/shiwenv"):
            links.append(base_url + href)
    return list(set(links))

def get_poem_data(poem_url):
    """
    获取每首诗的详细信息：标题、朝代、作者、正文、译文
    """
    response = requests.get(poem_url, headers=headers)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, "html.parser")

    # 提取标题
    title_tag = soup.find("h1")
    title = title_tag.text.strip() if title_tag else ""

    # 提取朝代和作者信息
    source = soup.find("p", class_="source")
    dynasty = ""
    author = ""
    if source:
        a_tags = source.find_all("a")
        if len(a_tags) >= 2:
            dynasty = a_tags[1].text.strip()
            author = a_tags[0].text.strip()

    dynasty = re.sub(r"[〔〕]", "", dynasty)

    # 提取正文内容
    content_div = soup.find("div", class_="contson")
    content = content_div.get_text(separator="\n").strip() if content_div else ""

    # 提取译文
    translation = ""
    translation_div = soup.find("div", class_="contyishang")
    if translation_div:
        translation = translation_div.get_text(separator="\n").strip()

    return {
        "title": title,
        "dynasty": dynasty,
        "author": author,
        "content": content,
        "translation": translation
    }

def main():
    poem_links = get_poem_links()
    print(f"共找到 {len(poem_links)} 首诗的链接。")

    all_poems = []
    for index, link in enumerate(poem_links, 1):
        try:
            data = get_poem_data(link)
            all_poems.append(data)
            print(f"({index}/{len(poem_links)}) 爬取：{data['title']} - 翻译：{'有' if data['translation'] else '无'}")
            time.sleep(1)
        except Exception as e:
            print(f"爬取出错：{link}，错误信息：{e}")

    with open("chuzhong_poems.json", "w", encoding="utf-8") as f:
        json.dump(all_poems, f, ensure_ascii=False, indent=2)

    print("爬取完成，数据已保存为 chuzhong_poems.json。")

if __name__ == "__main__":
    main()


In [ ]:
#清洗初中古诗文
import re
import json

def clean_content(text):
    """
    对 content 字段进行清洗：
    1. 删除全角括号（……）和半角括号 (...) 内的内容；
    2. 按句号、叹号、冒号和换行符分句，去除空白句子；
    3. 将分句后的内容合并成一个连续字符串。
    """
    text = re.sub(r"（[^）]*）", "", text)
    text = re.sub(r"\([^)]*\)", "", text)
    sentences = re.split(r'[。！？：\n]', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return "".join(sentences)

def clean_translation(text):
    """
    对 translation 字段进行清洗：
    1. 删除“译文及注释\n\n\n\n\n\n\n译文\n”前缀；
    2. 删除“注释”及其后内容；
    3. 按标点分句并去空白；
    4. 合并成一个连续字符串。
    """
    prefix_pattern = r"^译文及注释\s*\n+\s*译文\s*\n+"
    text = re.sub(prefix_pattern, "", text)
    if "注释" in text:
        text = text.split("注释")[0]
    sentences = re.split(r'[。！？：\n]', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return "".join(sentences)

def contains_english(value):
    """
    判断字符串或列表中是否包含英文字母
    """
    if isinstance(value, str):
        return bool(re.search(r'[A-Za-z]', value))
    elif isinstance(value, list):
        return any(re.search(r'[A-Za-z]', item) for item in value)
    return False

def main():
    input_file = "chuzhong_poems.json"
    output_file = "chuzhong_poems_clean.json"

    with open(input_file, "r", encoding="utf-8") as f:
        poems = json.load(f)

    original_count = len(poems)
    cleaned_poems = []

    for poem in poems:
        if "content" in poem:
            poem["content"] = clean_content(poem["content"])
        if "translation" in poem:
            poem["translation"] = clean_translation(poem["translation"])

        skip = False
        for field in ["title", "dynasty", "author", "content", "translation"]:
            if field in poem and re.search("展开阅读全文", poem[field]):
                skip = True
                break
            if field in poem and contains_english(poem[field]):
                skip = True
                break
        if skip:
            continue

        cleaned_poems.append(poem)

    cleaned_count = len(cleaned_poems)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(cleaned_poems, f, ensure_ascii=False, indent=4)

    print(f"原始数据数量: {original_count}")
    print(f"清洗后数据数量: {cleaned_count}")

if __name__ == "__main__":
    main()


In [ ]:
#爬取高中古诗文
import requests
from bs4 import BeautifulSoup
import json
import time
import re

base_url = "https://www.gushiwen.cn"
list_url = f"{base_url}/gushi/gaozhong.aspx"
headers = {"User-Agent": "Mozilla/5.0"}

def get_poem_links():
    response = requests.get(list_url, headers=headers)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith("/shiwenv"):
            links.append(base_url + href)
    return list(set(links))

def get_poem_data(poem_url):
    response = requests.get(poem_url, headers=headers)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, "html.parser")

    title_tag = soup.find("h1")
    title = title_tag.text.strip() if title_tag else ""

    source = soup.find("p", class_="source")
    dynasty = ""
    author = ""
    if source:
        a_tags = source.find_all("a")
        if len(a_tags) >= 2:
            dynasty = a_tags[1].text.strip()
            author = a_tags[0].text.strip()

    dynasty = re.sub(r"[〔〕]", "", dynasty)

    content_div = soup.find("div", class_="contson")
    content = content_div.get_text(separator="\n").strip() if content_div else ""

    translation = ""
    translation_div = soup.find("div", class_="contyishang")
    if translation_div:
        translation = translation_div.get_text(separator="\n").strip()

    return {
        "title": title,
        "dynasty": dynasty,
        "author": author,
        "content": content,
        "translation": translation
    }

def main():
    poem_links = get_poem_links()
    print(f"共找到 {len(poem_links)} 首高中古诗链接。")

    all_poems = []
    for idx, link in enumerate(poem_links, 1):
        try:
            data = get_poem_data(link)
            all_poems.append(data)
            print(f"({idx}/{len(poem_links)}) 爬取：{data['title']} - 翻译：{'有' if data['translation'] else '无'}")
            time.sleep(1)
        except Exception as e:
            print(f"爬取失败：{link}，错误：{e}")

    with open("gaozhong_poems.json", "w", encoding="utf-8") as f:
        json.dump(all_poems, f, ensure_ascii=False, indent=2)

    print("✅ 爬取完成，保存为 gaozhong_poems.json")

if __name__ == "__main__":
    main()


In [ ]:
#清洗高中古诗文
import re
import json

def clean_content(text):
    text = re.sub(r"（[^）]*）", "", text)
    text = re.sub(r"\([^)]*\)", "", text)
    sentences = re.split(r'[。！？：\n]', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return "".join(sentences)

def clean_translation(text):
    prefix_pattern = r"^译文及注释\s*\n+\s*译文\s*\n+"
    text = re.sub(prefix_pattern, "", text)
    if "注释" in text:
        text = text.split("注释")[0]
    sentences = re.split(r'[。！？：\n]', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return "".join(sentences)

def contains_english(value):
    if isinstance(value, str):
        return bool(re.search(r'[A-Za-z]', value))
    elif isinstance(value, list):
        return any(re.search(r'[A-Za-z]', item) for item in value)
    return False

def main():
    input_file = "gaozhong_poems.json"
    output_file = "gaozhong_poems_clean.json"

    with open(input_file, "r", encoding="utf-8") as f:
        poems = json.load(f)

    original_count = len(poems)
    cleaned_poems = []

    for poem in poems:
        if "content" in poem:
            poem["content"] = clean_content(poem["content"])
        if "translation" in poem:
            poem["translation"] = clean_translation(poem["translation"])

        skip = False
        for field in ["title", "dynasty", "author", "content", "translation"]:
            if field in poem and re.search("展开阅读全文", poem[field]):
                skip = True
                break
            if field in poem and contains_english(poem[field]):
                skip = True
                break
        if skip:
            continue

        cleaned_poems.append(poem)

    cleaned_count = len(cleaned_poems)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(cleaned_poems, f, ensure_ascii=False, indent=4)

    print(f"原始数据量: {original_count}")
    print(f"清洗后数据量: {cleaned_count}")

if __name__ == "__main__":
    main()


In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time
import re

base_url = "https://www.gushiwen.cn"
target_urls = [
    "/gushi/yongwu.aspx",
    "/gushi/xiatian.aspx",
    "/gushi/xiejing.aspx",
    "/gushi/qiutian.aspx",
    "/gushi/dongtian.aspx",
    "/gushi/yu.aspx",
    "/gushi/shuqing.aspx"
]
headers = {"User-Agent": "Mozilla/5.0"}

def get_poem_links_from_page(page_url):
    full_url = base_url + page_url
    response = requests.get(full_url, headers=headers)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith("/shiwenv"):
            links.append(base_url + href)
    return links

def get_poem_data(poem_url):
    response = requests.get(poem_url, headers=headers)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, "html.parser")

    title_tag = soup.find("h1")
    title = title_tag.text.strip() if title_tag else ""

    source = soup.find("p", class_="source")
    dynasty = ""
    author = ""
    if source:
        a_tags = source.find_all("a")
        if len(a_tags) >= 2:
            dynasty = a_tags[1].text.strip()
            author = a_tags[0].text.strip()

    dynasty = re.sub(r"[〔〕]", "", dynasty)

    content_div = soup.find("div", class_="contson")
    content = content_div.get_text(separator="\n").strip() if content_div else ""

    translation = ""
    translation_div = soup.find("div", class_="contyishang")
    if translation_div:
        translation = translation_div.get_text(separator="\n").strip()

    return {
        "title": title,
        "dynasty": dynasty,
        "author": author,
        "content": content,
        "translation": translation
    }

def main():
    all_links = set()
    for url in target_urls:
        links = get_poem_links_from_page(url)
        all_links.update(links)
        time.sleep(1)

    print(f"共收集到诗歌链接数：{len(all_links)}")

    poems = []
    for i, link in enumerate(all_links, 1):
        try:
            poem = get_poem_data(link)
            poems.append(poem)
            print(f"({i}/{len(all_links)}) 获取：{poem['title']} ✓")
            time.sleep(1)
        except Exception as e:
            print(f"获取失败：{link}，错误：{e}")

    with open("other_poems.json", "w", encoding="utf-8") as f:
        json.dump(poems, f, ensure_ascii=False, indent=2)

    print("✅ 爬取完成，保存为 other_poems.json")

if __name__ == "__main__":
    main()


In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import re
import time
from concurrent.futures import ThreadPoolExecutor

base_url = "https://www.gushiwen.cn"
all_urls = [
    "/gushi/xiaoxue.aspx",
    "/gushi/chuzhong.aspx",
    "/gushi/gaozhong.aspx",
    "/gushi/tangshi.aspx",
    "/gushi/songci.aspx",
    "/gushi/yongwu.aspx",
    "/gushi/xiatian.aspx",
    "/gushi/xiejing.aspx",
    "/gushi/qiutian.aspx",
    "/gushi/dongtian.aspx"
]

headers = {"User-Agent": "Mozilla/5.0"}

def get_poem_links_from_page(page_url):
    links = set()
    try:
        full_url = base_url + page_url
        res = requests.get(full_url, headers=headers, timeout=10)
        res.encoding = 'utf-8'
        soup = BeautifulSoup(res.text, "html.parser")
        for a in soup.find_all("a", href=True):
            if a["href"].startswith("/shiwenv"):
                links.add(base_url + a["href"])
    except Exception as e:
        print(f"❌ 页面出错：{page_url} | 错误: {e}")
    return links

def get_poem_data(poem_url):
    try:
        res = requests.get(poem_url, headers=headers, timeout=10)
        res.encoding = 'utf-8'
        soup = BeautifulSoup(res.text, "html.parser")
        title = soup.find("h1").text.strip() if soup.find("h1") else ""
        source = soup.find("p", class_="source")
        dynasty, author = "", ""
        if source:
            a_tags = source.find_all("a")
            if len(a_tags) >= 2:
                dynasty = a_tags[1].text.strip()
                author = a_tags[0].text.strip()

        dynasty = re.sub(r"[〔〕]", "", dynasty)
        
        content_div = soup.find("div", class_="contson")
        content = content_div.get_text(separator="\n").strip() if content_div else ""
        translation_div = soup.find("div", class_="contyishang")
        translation = translation_div.get_text(separator="\n").strip() if translation_div else ""
        return {
            "title": title,
            "dynasty": dynasty,
            "author": author,
            "content": content,
            "translation": translation
        }
    except Exception as e:
        print(f"❌ 获取失败：{poem_url} | 错误: {e}")
        return None

def clean_content(text):
    text = re.sub(r"（[^）]*）", "", text)
    text = re.sub(r"\([^)]*\)", "", text)
    sentences = re.split(r'[。！？：\n]', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return "".join(sentences)

def clean_translation(text):
    prefix_pattern = r"^译文及注释\s*\n+\s*译文\s*\n+"
    text = re.sub(prefix_pattern, "", text)
    if "注释" in text:
        text = text.split("注释")[0]
    sentences = re.split(r'[。！？：\n]', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return "".join(sentences)

def contains_english(value):
    if isinstance(value, str):
        return bool(re.search(r'[A-Za-z]', value))
    elif isinstance(value, list):
        return any(re.search(r'[A-Za-z]', item) for item in value)
    return False

def clean_poems(poems):
    cleaned_poems = []
    for poem in poems:
        if not poem:
            continue
        poem["content"] = clean_content(poem["content"]) if "content" in poem else ""
        poem["translation"] = clean_translation(poem["translation"]) if "translation" in poem else ""
        skip = False
        for field in ["title", "dynasty", "author", "content", "translation"]:
            if field in poem:
                if "展开阅读全文" in poem[field] or contains_english(poem[field]):
                    skip = True
                    break
        if not skip:
            cleaned_poems.append(poem)
    return cleaned_poems

def main():
    all_links = set()
    for url in all_urls:
        all_links.update(get_poem_links_from_page(url))
        time.sleep(0.5)  # 避免请求过快被封 IP

    print(f"✅ 共收集诗歌链接数：{len(all_links)}")

    with ThreadPoolExecutor(max_workers=10) as executor:
        poems = list(executor.map(get_poem_data, list(all_links)))

    with open("poems.json", "w", encoding="utf-8") as f:
        json.dump(poems, f, ensure_ascii=False, indent=2)
    print("📦 已保存原始数据到 poems.json")

    cleaned = clean_poems(poems)
    with open("poems_clean.json", "w", encoding="utf-8") as f:
        json.dump(cleaned, f, ensure_ascii=False, indent=2)
    print("🧼 清洗完成，保存为 poems_clean.json")
    print(f"✨ 原始数量: {len(poems)}，清洗后数量: {len(cleaned)}")

if __name__ == "__main__":
    main()


In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import re
import time
from concurrent.futures import ThreadPoolExecutor

base_url = "https://www.gushiwen.cn"
all_urls = [
    "/gushi/xiaoxue.aspx",
    "/gushi/chuzhong.aspx",
    "/gushi/gaozhong.aspx",
    "/gushi/tangshi.aspx",
    "/gushi/songci.aspx",
    "/gushi/yongwu.aspx",
    "/gushi/xiatian.aspx",
    "/gushi/xiejing.aspx",
    "/gushi/qiutian.aspx",
    "/gushi/dongtian.aspx"
]

headers = {"User-Agent": "Mozilla/5.0"}

def get_poem_links_from_page(page_url):
    links = set()
    try:
        full_url = base_url + page_url
        res = requests.get(full_url, headers=headers, timeout=10)
        res.encoding = 'utf-8'
        soup = BeautifulSoup(res.text, "html.parser")
        for a in soup.find_all("a", href=True):
            if a["href"].startswith("/shiwenv"):
                links.add(base_url + a["href"])
    except Exception as e:
        print(f"❌ 页面出错：{page_url} | 错误: {e}")
    return links

def get_poem_data(poem_url):
    try:
        res = requests.get(poem_url, headers=headers, timeout=10)
        res.encoding = 'utf-8'
        soup = BeautifulSoup(res.text, "html.parser")
        title = soup.find("h1").text.strip() if soup.find("h1") else ""
        source = soup.find("p", class_="source")
        dynasty, author = "", ""
        if source:
            a_tags = source.find_all("a")
            if len(a_tags) >= 2:
                dynasty = a_tags[1].text.strip()
                author = a_tags[0].text.strip()

        dynasty = re.sub(r"[〔〕]", "", dynasty)
        
        content_div = soup.find("div", class_="contson")
        content = content_div.get_text(separator="\n").strip() if content_div else ""
        translation_div = soup.find("div", class_="contyishang")
        translation = translation_div.get_text(separator="\n").strip() if translation_div else ""
        return {
            "title": title,
            "dynasty": dynasty,
            "author": author,
            "content": content,
            "translation": translation
        }
    except Exception as e:
        print(f"❌ 获取失败：{poem_url} | 错误: {e}")
        return None

def clean_content(text):
    text = re.sub(r"（[^）]*）", "", text)  # 清除全角括号中的内容
    text = re.sub(r"\([^)]*\)", "", text)   # 清除半角括号中的内容
    sentences = re.split(r'[。！？：\n]', text)  # 根据标点分句
    sentences = [s.strip() for s in sentences if s.strip()]  # 去除空句子
    return "".join(sentences)

def clean_translation(text):
    prefix_pattern = r"^译文及注释\s*\n+\s*译文\s*\n+"  # 清除前缀
    text = re.sub(prefix_pattern, "", text)
    if "注释" in text:
        text = text.split("注释")[0]  # 去除“注释”及其后的内容
    sentences = re.split(r'[。！？：\n]', text)  # 根据标点分句
    sentences = [s.strip() for s in sentences if s.strip()]  # 去除空句子
    return "".join(sentences)

def contains_english(value):
    if isinstance(value, str):
        return bool(re.search(r'[A-Za-z]', value))  # 判断是否包含英文字符
    elif isinstance(value, list):
        return any(re.search(r'[A-Za-z]', item) for item in value)
    return False

def clean_poems(poems):
    cleaned_poems = []
    for poem in poems:
        if not poem:
            continue
        poem["content"] = clean_content(poem["content"]) if "content" in poem else ""
        poem["translation"] = clean_translation(poem["translation"]) if "translation" in poem else ""
        
        # 清洗 translation 如果为空
        if 'translation' in poem and not poem['translation']:
            del poem['translation']
        
        skip = False
        for field in ["title", "dynasty", "author", "content", "translation"]:
            if field in poem:
                if "展开阅读全文" in poem[field] or contains_english(poem[field]):  # 过滤含有“展开阅读全文”或英文的诗歌
                    skip = True
                    break
        if not skip:
            cleaned_poems.append(poem)
    return cleaned_poems

def main():
    all_links = set()
    for url in all_urls:
        all_links.update(get_poem_links_from_page(url))
        time.sleep(0.5)  # 避免请求过快被封 IP

    print(f"✅ 共收集诗歌链接数：{len(all_links)}")

    with ThreadPoolExecutor(max_workers=10) as executor:
        poems = list(executor.map(get_poem_data, list(all_links)))

    with open("poems.json", "w", encoding="utf-8") as f:
        json.dump(poems, f, ensure_ascii=False, indent=2)
    print("📦 已保存原始数据到 poems.json")

    cleaned = clean_poems(poems)
    with open("poems_clean.json", "w", encoding="utf-8") as f:
        json.dump(cleaned, f, ensure_ascii=False, indent=2)
    print("🧼 清洗完成，保存为 poems_clean.json")
    print(f"✨ 原始数量: {len(poems)}，清洗后数量: {len(cleaned)}")

if __name__ == "__main__":
    main()
